<a href="https://colab.research.google.com/github/Dilandds/CNN-emotion-detection/blob/main/CCN_emotion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# check gpu
import torch

print(torch.__version__)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# kaggle no longer gives a downloadable json, paste creds instead
import json, os
from getpass import getpass

username = input("kaggle username: ")
key = getpass("kaggle key: ")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": username, "key": key}, f)

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d /content/fer2013

In [ ]:
# sanity check - counts per class
import os

DATA_DIR = "/content/fer2013"

for split in ["train", "test"]:
    split_dir = os.path.join(DATA_DIR, split)
    classes = sorted(os.listdir(split_dir))
    print(f"\n{split} ({len(classes)} classes)")
    total = 0
    for c in classes:
        n = len(os.listdir(os.path.join(split_dir, c)))
        total += n
        print(f"  {c:10s} {n}")
    print(f"  total: {total}")

In [ ]:
# look at some actual images
import matplotlib.pyplot as plt
from PIL import Image
import random

classes = sorted(os.listdir(os.path.join(DATA_DIR, "train")))

fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for i, c in enumerate(classes):
    folder = os.path.join(DATA_DIR, "train", c)
    files = os.listdir(folder)
    for row in range(2):
        img = Image.open(os.path.join(folder, random.choice(files)))
        axes[row, i].imshow(img, cmap="gray")
        axes[row, i].axis("off")
        if row == 0:
            axes[row, i].set_title(c)
plt.tight_layout()
plt.show()

In [ ]:
# check size / mode of a random image
sample_path = os.path.join(DATA_DIR, "train", "happy", os.listdir(os.path.join(DATA_DIR, "train", "happy"))[0])
img = Image.open(sample_path)
print("size:", img.size)
print("mode:", img.mode)

In [ ]:
# class balance
counts = {c: len(os.listdir(os.path.join(DATA_DIR, "train", c))) for c in classes}

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values())
plt.title("train class balance")
plt.ylabel("num images")
plt.show()

counts

In [ ]:
# custom dataset - just scans the folders and builds a (path, label) list
from torch.utils.data import Dataset

class FERDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.transform = transform
        self.classes = classes
        self.samples = []
        for label, c in enumerate(classes):
            folder = os.path.join(root_dir, c)
            for fname in os.listdir(folder):
                self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("L")  # grayscale
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# transforms - basic normalize for val/test, some augmentation for train
from torchvision import transforms

train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

eval_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

In [ ]:
# split train folder into train/val, keep test folder as is
from torch.utils.data import random_split, DataLoader

full_train = FERDataset(os.path.join(DATA_DIR, "train"), classes, transform=train_tfms)
test_ds = FERDataset(os.path.join(DATA_DIR, "test"), classes, transform=eval_tfms)

val_size = int(0.1 * len(full_train))
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(full_train, [train_size, val_size])

# val should use eval transforms, not the augmented ones
val_ds.dataset = FERDataset(os.path.join(DATA_DIR, "train"), classes, transform=eval_tfms)

print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# sanity check one batch
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

In [ ]:
# cnn - conv/bn/pool x3 then fc down to 7 classes
import torch.nn as nn

class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

        # 48 -> 24 -> 12 -> 6 after 3 pools
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
# sanity check - dummy input, confirm output is (batch, 7)
model = EmotionCNN().to(device)
dummy = torch.randn(1, 1, 48, 48).to(device)
out = model(dummy)
print(out.shape)

In [ ]:
# save checkpoints to drive so they survive if colab disconnects
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = "/content/drive/MyDrive/fer2013_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# one pass over the data, updates weights if train=True
def run_epoch(loader, train=True):
    model.train() if train else model.eval()

    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
EPOCHS = 20

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"epoch {epoch+1}/{EPOCHS} - train loss {train_loss:.3f} acc {train_acc:.3f} | val loss {val_loss:.3f} acc {val_acc:.3f}")

    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, f"model_epoch{epoch+1}.pt"))

In [ ]:
# final test accuracy
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"test loss {test_loss:.3f}  test acc {test_acc:.3f}")

In [ ]:
# confusion matrix - which emotions get mixed up with which
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues")
plt.show()

In [ ]:
# a few example predictions - mix of correct and wrong, different ones each run
idxs = random.sample(range(len(test_ds)), 16)
images = torch.stack([test_ds[i][0] for i in idxs])
labels = torch.tensor([test_ds[i][1] for i in idxs])

images_dev = images.to(device)
preds = model(images_dev).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = images[i].squeeze().numpy() * 0.5 + 0.5  # undo normalize
    ax.imshow(img, cmap="gray")
    correct = preds[i] == labels[i]
    ax.set_title(f"{classes[preds[i]]}", color="green" if correct else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# loss/accuracy curves
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], label="train")
axes[0].plot(epochs_range, history["val_loss"], label="val")
axes[0].set_title("loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="train")
axes[1].plot(epochs_range, history["val_acc"], label="val")
axes[1].set_title("accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()



- built a small CNN from scratch (3 conv/bn/pool blocks + 2 fc) instead of using a pretrained
  backbone, since images are only 48x48 grayscale and dataset is small
- disgust class is heavily underrepresented (436 vs 7215 for happy) - shows up as the weakest
  row in the confusion matrix, model tends to confuse it with angry/fear
- used dropout + light augmentation (flip, small rotation) as the main defense against
  overfitting given how small this dataset is
- with more time: class weighting or oversampling for disgust, try a deeper/pretrained backbone,
  maybe a learning rate scheduler, and the temporal self-supervised extension (video sequences
  instead of single frames - discuss verbally, not implemented here)